# 05 — MCP : historique, tickets et validation humaine

**Objectif :** comprendre et tester le serveur MCP MaintiDoc.

MCP est un protocole client-serveur. Le serveur expose des **outils** typés ; le
client (l'agent ou l'application) les appelle. Le modèle ne reçoit pas les
identifiants de la base : il voit seulement le schéma des outils.

Trois outils sont exposés :

1. `get_equipment_history(equipment_id)` ;
2. `get_open_tickets(equipment_id)` ;
3. `create_ticket(..., approval_token, idempotency_key)`.


## Étape 1 — Lire le serveur

Le serveur est dans `src/maintidoc/mcp_server.py`. On affiche son code pour voir
la déclaration des outils et les fonctions appelées.


In [1]:
from pathlib import Path

source = Path("../src/maintidoc/mcp_server.py").read_text(encoding="utf-8")
print(source)

"""Serveur MCP MaintiDoc : historique, tickets ouverts, création contrôlée."""
from __future__ import annotations

from mcp.server.mcpserver import MCPServer

from .mcp_store import (
    create_ticket as store_create_ticket,
    get_equipment_history as store_get_history,
    get_open_tickets as store_get_open_tickets,
)

mcp = MCPServer(
    name="MaintiDoc SAN",
    instructions=(
        "Serveur documentaire MaintiDoc. Les outils lisent un historique fictif "
        "et créent un ticket uniquement avec un jeton d'approbation humaine."
    ),
)


@mcp.tool()
def get_equipment_history(equipment_id: str) -> dict:
    """Retourne l'historique des interventions d'un équipement SAN."""
    return store_get_history(equipment_id)


@mcp.tool()
def get_open_tickets(equipment_id: str) -> dict:
    """Retourne les tickets ouverts d'un équipement SAN."""
    return store_get_open_tickets(equipment_id)


@mcp.tool()
def create_ticket(equipment_id: str, title: str, description: str,
          

### Lecture du résultat

On voit `MCPServer`, les décorateurs `@mcp.tool()` et les trois fonctions. Le
serveur ne contient pas de logique métier : il délègue à `mcp_store.py`.


## Étape 2 — Lister les outils

Le client lance le serveur en sous-processus via `stdio_client` : le serveur
lit/écrit du JSON-RPC sur l'entrée/sortie standard.

La fonction `list_tools()` encapsule ce transport pour éviter de gérer une
session asynchrone dans chaque cellule.


In [2]:
from maintidoc.mcp_calls import list_tools

for tool in list_tools():
    print(tool["name"], "—", tool["description"])

get_equipment_history — Retourne l'historique des interventions d'un équipement SAN.
get_open_tickets — Retourne les tickets ouverts d'un équipement SAN.
create_ticket — Crée un ticket après vérification du jeton d'approbation humaine.

    Le jeton est lié au contenu exact (équipement, titre, description).
    


### Lecture du résultat

Les trois outils doivent apparaître avec leur description et leur schéma
d'entrée. Le serveur est prêt à recevoir des appels.


## Étape 3 — Consulter l'historique

On appelle `get_equipment_history` avec `SAN-LAB-2000`.


In [3]:
from maintidoc.mcp_calls import call_tool

history = call_tool("get_equipment_history", {"equipment_id": "SAN-LAB-2000"})
print(history)

{'ok': True, 'equipment_id': 'SAN-LAB-2000', 'entries': [{'ticket': 'INC-104', 'date': '2026-09-18', 'asset': 'Fabric A', 'summary': 'Perte temporaire de chemin HBA1 ; surveillance', 'status': 'clos'}, {'ticket': 'INC-117', 'date': '2026-10-02', 'asset': 'Disque B03', 'summary': 'Pré-alerte ; examen du groupe de protection', 'status': 'clos'}, {'ticket': 'INC-125', 'date': '2026-10-20', 'asset': 'Volume VOL-DB', 'summary': 'Capacité à 88 % ; analyse de tendance', 'status': 'ouvert'}, {'ticket': 'INC-130', 'date': '2026-10-25', 'asset': 'Contrôleur B', 'summary': 'Événement non reproductible ; suivi', 'status': 'ouvert'}]}


### Lecture du résultat

L'historique contient quatre interventions fictives (`INC-104`, `INC-117`,
`INC-125`, `INC-130`). Chaque appel est journalisé dans `data/mock/audit.jsonl`.


## Étape 4 — Préparer et approuver un ticket

La création se fait en deux temps :

1. `prepare_ticket()` crée un brouillon local, sans action externe ;
2. `approve_ticket()` simule l'approbation humaine et génère un jeton lié au
   contenu.


In [4]:
from maintidoc.mcp_store import approve_ticket, prepare_ticket

draft = prepare_ticket(
    equipment_id="SAN-LAB-2000",
    title="Alerte AL-01 sur fabric A",
    description="Chemin FC dégradé sur HBA1, récurrence à vérifier.",
)["draft"]

approval = approve_ticket(draft["draft_id"])
print("Draft :", draft["draft_id"])
print("Token :", approval["approval_token"])

Draft : DRAFT-f730253d
Token : APPROVAL-b7feea52d69e


### Lecture du résultat

Le brouillon contient un hash du contenu. Le jeton d'approbation est lié à ce
hash : si le titre ou la description change, le jeton ne sera plus valide.


## Étape 5 — Créer le ticket via MCP

On appelle `create_ticket` avec le jeton. Le serveur vérifie :

- que le jeton existe ;
- que le contenu correspond au hash approuvé ;
- qu'aucun ticket avec la même clé d'idempotence n'existe déjà.


In [5]:
created = call_tool(
    "create_ticket",
    {
        "equipment_id": "SAN-LAB-2000",
        "title": draft["content"]["title"],
        "description": draft["content"]["description"],
        "approval_token": approval["approval_token"],
        "idempotency_key": "demo-al01-001",
    },
)
print(created)

{'ok': True, 'ticket': {'ticket_id': 'TCK-72CAF2', 'equipment_id': 'SAN-LAB-2000', 'title': 'Alerte AL-01 sur fabric A', 'description': 'Chemin FC dégradé sur HBA1, récurrence à vérifier.', 'status': 'open', 'created_at': '2026-11-26T19:17:09.307294+00:00', 'approved_by': 'technicien-habilite', 'approval_token': 'APPROVAL-b7feea52d69e', 'idempotency_key': 'demo-al01-001'}, 'idempotent': False}


### Lecture du résultat

Le ticket est créé avec un identifiant `TCK-...`. Le champ `idempotent` vaut
`False` au premier appel.


## Étape 6 — Tester l'idempotence et le refus

Deux appels avec la même clé doivent retourner le même ticket. Un jeton invalide
doit être refusé.


In [6]:
again = call_tool(
    "create_ticket",
    {
        "equipment_id": "SAN-LAB-2000",
        "title": draft["content"]["title"],
        "description": draft["content"]["description"],
        "approval_token": approval["approval_token"],
        "idempotency_key": "demo-al01-001",
    },
)
print("Idempotent :", again["idempotent"])

denied = call_tool(
    "create_ticket",
    {
        "equipment_id": "SAN-LAB-2000",
        "title": "Sans approbation",
        "description": "Ne doit pas être créé.",
        "approval_token": "INVALID",
    },
)
print("Refus :", denied)

Idempotent : True


Refus : {'ok': False, 'error': "Jeton d'approbation absent ou inconnu"}


### Lecture du résultat

Le deuxième appel renvoie le même ticket avec `idempotent: True`. Le jeton
invalide renvoie `ok: False` et aucun ticket n'est créé.


## Étape 7 — Journal d'audit

Chaque appel est tracé. On lit les dernières lignes de `audit.jsonl`.


In [7]:
from maintidoc.config import MOCK_DIR

audit = (MOCK_DIR / "audit.jsonl").read_text(encoding="utf-8").splitlines()
for line in audit[-6:]:
    print(line)

{"at": "2026-11-26T19:17:07.590276+00:00", "action": "get_equipment_history", "equipment_id": "SAN-LAB-2000", "count": 4}
{"at": "2026-11-26T19:17:07.750744+00:00", "action": "prepare_ticket", "draft_id": "DRAFT-f730253d", "equipment_id": "SAN-LAB-2000"}
{"at": "2026-11-26T19:17:07.763746+00:00", "action": "approve_ticket", "draft_id": "DRAFT-f730253d", "approver": "technicien-habilite"}
{"at": "2026-11-26T19:17:09.309295+00:00", "action": "create_ticket", "ticket_id": "TCK-72CAF2", "equipment_id": "SAN-LAB-2000", "approved_by": "technicien-habilite"}


### Lecture du résultat

On voit les actions `get_equipment_history`, `prepare_ticket`, `approve_ticket`,
`create_ticket`, avec les identifiants et les horodatages. C'est la traçabilité
minimale attendue.


## Bilan MCP

- Trois outils typés.
- Un jeton d'approbation lié au contenu.
- Une idempotence.
- Un journal d'audit.
- Un client de test.

**Limites :** pas encore d'authentification, pas de RBAC, pas de vraie base, pas
de LangGraph. Ce sont les étapes suivantes du CDC.
